In [21]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
import json, random
from datasets import Dataset
import os

In [22]:

# 一些参数
model_path = '/root/autodl-tmp/self-llm/model/Qwen/Qwen2___5-0___5B-Instruct'
# model_name = "/qwen2.5-0.5-instruct/"  # 模型名或者本地路径
train_data_path = '../../dataset/prep/1/train.json'
test_file_path = '../../dataset/prep/1/test.json'
res_path = './../dataset/res/1/res.json'
save_path = "./output/save_model_7b/"

num_train_epochs=1
per_device_train_batch_size=1
per_device_eval_batch_size=1
warmup_steps=10
weight_decay=0.01
logging_steps=1
use_cpu=False

# 创建标签到索引的映射
label_to_id = {
    "胸痹心痛病": 0,
    "心衰病": 1,
    "眩晕病": 2,
    "心悸病": 3
}

num_labels = len(label_to_id)  # 根据你的标签数量设置num_labels

# 读取jsonl文件
def read_jsonl(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return data

data = read_jsonl(train_data_path)
random.shuffle(data)

# 将文本标签转换为数值标签
for example in data:
    example['label'] = label_to_id[example['output']]

# 检查标签范围
for example in data:
    assert 0 <= example['label'] < len(label_to_id), f"Label out of range: {example['output']}" 

# 将数据转换为datasets库的Dataset对象
dataset = Dataset.from_list(data)

# 将数据集拆分为训练集和验证集
dataset = dataset.train_test_split(test_size=0.2)

In [23]:
# 加载预训练的 Qwen2 模型和分词器
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path, num_labels=num_labels)  
print(model)
model.config.pad_token_id = 151643  # 定义pad token，模型才会忽略后面那些pad而是把真正最后一个token的hidden state用于分类


Some weights of Qwen2ForSequenceClassification were not initialized from the model checkpoint at /root/autodl-tmp/self-llm/model/Qwen/Qwen2___5-0___5B-Instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Qwen2ForSequenceClassification(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2SdpaAttention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
          (rotary_emb): Qwen2RotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): 

In [24]:



# 定义一个函数来处理数据集中的文本
def preprocess_function(examples):
    return tokenizer(examples['input'], truncation=True, padding=True, return_tensors="pt")

# 对数据集进行预处理
encoded_dataset = dataset.map(preprocess_function, batched=True)


Map:   0%|          | 0/640 [00:00<?, ? examples/s]

Map:   0%|          | 0/160 [00:00<?, ? examples/s]

In [25]:

# 定义训练参数
training_args = TrainingArguments(
    output_dir=save_path,                           # 输出目录
    num_train_epochs=num_train_epochs,              # 训练的epoch数
    per_device_train_batch_size=per_device_train_batch_size,    # 每个设备的训练batch size
    per_device_eval_batch_size=per_device_eval_batch_size,      # 每个设备的评估batch size
    warmup_steps=warmup_steps,                  # 预热步数
    weight_decay=weight_decay,                  # 权重衰减
    logging_dir=save_path,                      # 日志目录
    logging_steps=logging_steps,
    evaluation_strategy="epoch",
    save_strategy="epoch",    # 每个epoch保存一次检查点
    save_total_limit=3,       # 最多保存3个检查点，旧的会被删除
    use_cpu=False
)

# 定义Trainer
trainer = Trainer(
    model=model,                                    # 模型
    args=training_args,                             # 训练参数
    train_dataset=encoded_dataset['train'],         # 训练数据集
    eval_dataset=encoded_dataset['test']            # 评估数据集
)

# 打印训练集和验证集中的一些样本
print("Train dataset sample:")
print(encoded_dataset['train'][0])  # 打印训练集中的第一个样本

print("Eval dataset sample:")
print(encoded_dataset['test'][0])  # 打印验证集中的第一个样本

# 开始训练
trainer.train()
trainer.save_state()
trainer.save_model(output_dir=save_path)
tokenizer.save_pretrained(save_path)

/root/miniconda3/lib/python3.10/site-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Train dataset sample:
{'instruction': '\n任务：根据患者[基本信息],[主诉],[症状],[中医望闻切诊],[病史],[体格检查],[辅助检查]等信息,输出患者疾病类别\n', 'input': "\n[基本信息]:患者性别为男,职业为职员,年龄为32岁,婚姻为已婚,发病节气在大雪。\n[主诉]:主  诉：阵发性头晕头痛1月，加重2日。\n[症状]:阵发性头晕头痛，伴头胀，休息可缓解，易紧张，左眼胀，无胸闷心慌，手心易出汗，无乏力气短，纳可，眠一般，夜尿2次，大便调。\n[中医望闻切诊]:中医望闻切诊：表情自然，面色红润，形体正常，动静姿态，语气清，气息平；无异常气味，舌红、苔薄白，舌下络脉无异常，脉数。\n[病史]:现病史，患者于1月前无明显诱因出现阵发性头晕头痛，查体血压145/92mmHg，诊为高血压，未予治疗，2日前患者头晕加重，最高血压达197/109mmHg，服用厄贝沙坦效不佳，为求进一步中西医结合治疗，入住我院区，入院症见，既往史，既往身体健康状况可，否认心脏病、否认糖尿病等慢性疾病病史，否认肝炎、否认结核等传染病史，预防接种史不详，否认手术史、否认重大外伤史，否认输血史，否认药物过敏史、否认其他接触物过敏史，个人史，久居本地，无疫水、疫源接触史，无嗜酒史，吸烟史10余年，平均8支/天，无放射线物质接触史，否认麻醉毒品等嗜好，否认冶游史，否认食物过敏史，否认传染病史，婚育史，适龄婚育，育有1子，配偶及儿子体健，家族史，哥哥1人，父母及哥哥体健，否认家族性遗传病史。\n[体格检查]:生命体征体温：37.3℃ 脉搏：98次/分 呼吸：17次/分 血压：182/109mmHg Padua评分：0分 低危 卒中风险评估：中危。一般情况：患者青年男性，发育正常，营养良好，神志清楚，步入病房，查体合作，皮肤黏膜：全身皮肤及粘膜无黄染，未见皮下出血，淋巴结浅表淋巴结未及肿大。标题定位符头颅五官无畸形，眼睑无水肿，巩膜无黄染，双侧瞳孔等大等圆，对光反射灵敏，外耳道无异常分泌物，鼻外观无畸形，口唇红润，伸舌居中，双侧扁桃体正常，表面未见脓性分泌物，标题定位符颈软，无抵抗感，双侧颈静脉正常，气管居中，甲状腺未及肿大，未闻及血管杂音。标题定位符胸廓正常，双肺呼吸音清晰，未闻及干、湿罗音，未闻及胸膜摩擦音。心脏心界不大，心率98次/分，

Epoch,Training Loss,Validation Loss
1,3.360600,1.884148


We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)


('./output/save_model_7b/tokenizer_config.json',
 './output/save_model_7b/special_tokens_map.json',
 './output/save_model_7b/vocab.json',
 './output/save_model_7b/merges.txt',
 './output/save_model_7b/added_tokens.json',
 './output/save_model_7b/tokenizer.json')

# 测试

In [ ]:
# 读取jsonl文件
def read_jsonl(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return data

In [29]:
# 创建标签到索引的映射
label_to_id = {
    "胸痹心痛病": 0,
    "心衰病": 1,
    "眩晕病": 2,
    "心悸病": 3
}
# 创建 id_to_label 字典
id_to_label = {v: k for k, v in label_to_id.items()}

# 准备输入文本
texts = []
true_label = []
ID = []

data = read_jsonl(test_file_path)
for line in data:
    t = line['input']
    texts.append(t)
    # true_label.append(label_to_id[line['疾病']])
    ID.append(line['ID'])


KeyError: 'ID'

In [30]:
line

{'instruction': '\n任务：根据患者[基本信息],[主诉],[症状],[中医望闻切诊],[病史],[体格检查],[辅助检查]等信息,输出患者疾病类别\n',
 'input': "\n[基本信息]:患者性别为女,职业为退休,年龄为82岁,婚姻为丧偶,发病节气在小雪。\n[主诉]:主  诉：阵发性头晕2年余，加重1月余。\n[症状]:阵发性头晕，伴心慌，无头痛头胀，无胸闷胸痛，体力可，偶有反酸，心情焦虑，纳可，睡前一片安定，二便调。\n[中医望闻切诊]:中医望闻切诊：表情自然，面色少华，形体正常，动静姿态，语气清，气息平；无异常气味，舌红，苔少，有裂纹，舌下络脉无异常，脉弦细。\n[病史]:现病史，患者于2年前无明显诱因出现阵发性头晕，诊为高血压病，最高血压可达200/100mmHg，曾服用罗布麻控制血压，现服用伲福达，平素血压可控制在130/80mmHg，患者于1月前无明显诱因血压升高，血压波动较大，口服伲福达不能缓解，现为求进一步中西医结合专科诊疗，入住我病区，入院症见，既往史，既往腰椎间盘突出病史20余年，骨质疏松病史1年余，双眼青光眼病史6年余，否认糖尿病等慢性疾病病史，否认肝炎、否认结核等传染病史，预防接种史不详，曾于2014年，2018年分别行左右眼青光眼小梁切除术，否认重大外伤史，否认输血史，自述有B族维生素过敏史、否认其他接触物过敏史，个人史，久居本地，无疫水、疫源接触史，无嗜酒史，无吸烟史，无放射线物质接触史，否认麻醉毒品等嗜好，否认冶游史，自述有B族维生素过敏史，否认传染病史，婚育史，适龄婚育，育有1子1女，月经史，既往月经规律正常，现已绝经，家族史，否认家族性遗传病史。\n[体格检查]:生命体征体温：36.6℃ 脉搏：71次/分 呼吸：18次/分 血压：185/90mmHg VTE评分：1分  卒中风险评估：中危一般情况：患者，老年女性，发育正常，营养良好，神志清楚，查体合作，皮肤黏膜：全身皮肤及粘膜无黄染，未见皮下出血，淋巴结浅表淋巴结未及肿大。标题定位符头颅五官无畸形，眼睑无水肿，巩膜无黄染，双侧结膜充血，双侧瞳孔欠圆，对光反射灵敏，外耳道无异常分泌物，鼻外观无畸形，口唇红润，伸舌居中，双侧扁桃体正常，表面未见脓性分泌物，标题定位符颈软，无抵抗感，双侧颈静脉正常，气管居中，甲状腺未及肿大，未闻及血管杂音。标题定位符胸廓正常，双

In [ ]:
from transformers import Qwen2ForSequenceClassification, Qwen2Tokenizer
import torch
import json

# 加载模型和分词器
model_name = ""
tokenizer = Qwen2Tokenizer.from_pretrained(model_name)
model = Qwen2ForSequenceClassification.from_pretrained(model_name)
for parameter in model.parameters():
    parameter.requires_grad = False

# 对文本进行编码
inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")

# 进行推理
with torch.no_grad():
    outputs = model(**inputs)

# 获取预测结果
logits = outputs.logits
predictions = torch.argmax(logits, dim=-1)

results = []
for num, i in enumerate(predictions):
    results.append({"ID": ID[num], "疾病": id_to_label[i.item()]})

with open(res_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

# # 检查对应位置是否相同
# equal_positions = (predictions == torch.tensor(true_label)
# # 统计相同的位置数量
# num_equal = equal_positions.sum().item()

# acc = num_equal / len(true_label)
# print('ACC:{}'.format(acc))

